### Loading libraries and initializing useful functions

In [ ]:
import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import IntSlider, fixed, interact
from matplotlib.colors import ListedColormap, TwoSlopeNorm

%matplotlib inline

from config import IMAGE_DATA_DIR
from kuwahara_filters import classic_kuwahara, classic_kuwahara_flawed

In [ ]:
def load_and_resize(image_path, max_side=800):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR_RGB)
    h, w = img.shape[:2]

    if max(h, w) <= max_side:
        return img

    scale = max_side / float(max(h, w))
    new_w = int(w * scale)
    new_h = int(h * scale)

    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

In [ ]:
def add_gaussian_noise(image, sigma):
    if sigma == 0:
        return image.copy()

    noise = np.random.normal(0, sigma, image.shape)
    noisy_image = image.astype(np.float32) + noise

    return np.clip(noisy_image, 0, 255).astype(np.uint8)

In [ ]:
def update_kuwahara(image, noise_sigma, kuwahara_algorithm, **kuwahara_params):
    noisy_image = add_gaussian_noise(image, sigma=noise_sigma)
    denoised_image = kuwahara_algorithm(noisy_image, **kuwahara_params)

    fig, axes = plt.subplots(1, 3, figsize=(15, 7))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(noisy_image)
    axes[1].set_title(f"Noisy (sigma={noise_sigma})")
    axes[1].axis("off")

    axes[2].imshow(denoised_image)
    params = ",".join(
        "=".join(map(str, pair)) for pair in kuwahara_params.items()
    )
    axes[2].set_title(f"Filtered ({params})")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

### Artifacts in the naive RGB implementation

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "parrot.png", max_side=600)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 6))

correct = classic_kuwahara(image, radius=5)
flawed = classic_kuwahara_flawed(image, radius=5)
diff = diff = cv2.absdiff(correct, flawed)

axes[0].imshow(correct)
axes[0].set_title("Correct implementation")

axes[1].imshow(flawed)
axes[1].set_title("Flawed implementation")

axes[2].imshow(diff)
axes[2].set_title("Image difference")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y, x = 250, 300
h, w = 60, 60

crop_correct = correct[y : y + h, x : x + w]
crop_flawed = flawed[y : y + h, x : x + w]

img_with_box = image.copy()
cv2.rectangle(img_with_box, (x, y), (x + w, y + h), (255, 0, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_with_box)
axes[0].set_title("Original Image")

axes[1].imshow(crop_correct)
axes[1].set_title("Correct implementation (Zoom)")

axes[2].imshow(crop_flawed)
axes[2].set_title("Flawed implementation (Zoom)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Noise removal, stylization, and block artifacts

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "landscape.png", max_side=600)

In [ ]:
interact(
    update_kuwahara,
    image=fixed(image),
    kuwahara_algorithm=fixed(classic_kuwahara),
    noise_sigma=IntSlider(min=0, max=100, step=5, value=10),
    radius=IntSlider(min=1, max=10, step=1, value=1),
)

In [ ]:
noisy = add_gaussian_noise(image, sigma=15)
kuwahara_filtered = classic_kuwahara(noisy, radius=5)
gauss_filtered = cv2.GaussianBlur(
    src=noisy, ksize=(11, 11), sigmaX=5, borderType=cv2.BORDER_REPLICATE
)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0][0].imshow(image)
axes[0][0].set_title("Original")

axes[0][1].imshow(noisy)
axes[0][1].set_title("Noisy")

axes[1][0].imshow(gauss_filtered)
axes[1][0].set_title("Gaussian Blur")

axes[1][1].imshow(kuwahara_filtered)
axes[1][1].set_title("Kuwahara Filter")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
gray_img = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
gray_img = gray_img.astype(float) / 255.0
H, W = gray_img.shape

L = 11
kernel = np.ones((L, L), dtype=float) / (L * L)

# To ensure the Fourier transform of the filter matches the dimensions of
# the image's FFT, we must pad the small L x L kernel into an H x W matrix.
# For plotting purposes it's better to put it in the center.
padded_filter = np.zeros((H, W), dtype=float)
cy, cx = H // 2, W // 2
r = L // 2

y_start = cy - L // 2
x_start = cx - L // 2
padded_filter[y_start : y_start + L, x_start : x_start + L] = kernel

# Convolution (I * h)
conv_spatial = cv2.filter2D(gray_img, -1, kernel, borderType=cv2.BORDER_WRAP)

# Image Fourier Transform
fft_img = np.fft.fft2(gray_img)
fft_img_shifted = np.fft.fftshift(fft_img)
mag_img = np.log1p(np.abs(fft_img_shifted))

# Filter Fourier Transform
fft_filter = np.fft.fft2(padded_filter)
fft_filter_shifted = np.fft.fftshift(fft_filter)
mag_filter = np.abs(fft_filter_shifted)

# Spectrum multiplication (I * h in the frequency domain)
fft_product = fft_img * fft_filter
fft_product_shifted = np.fft.fftshift(fft_product)
mag_product = np.log1p(np.abs(fft_product_shifted))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

axes[0, 0].imshow(gray_img, cmap="gray")
axes[0, 0].set_title("1. Original $I(x,y)$", fontsize=14)

axes[0, 1].imshow(padded_filter, cmap="gray")
axes[0, 1].set_title("2. Filter $h(x,y)$", fontsize=14)

axes[0, 2].imshow(conv_spatial, cmap="gray")
axes[0, 2].set_title("3. Convolution $(I * h)(x,y)$", fontsize=14)

axes[1, 0].imshow(mag_img, cmap="magma")
axes[1, 0].set_title("4. Image Spectrum $|\mathcal{F}(I)|$ (log)", fontsize=14)

axes[1, 1].imshow(mag_filter, cmap="gray")
axes[1, 1].set_title("5. Filter Spectrum $|\mathcal{F}(h)|$", fontsize=14)

axes[1, 2].imshow(mag_product, cmap="magma")
axes[1, 2].set_title(
    "6. Product $|\mathcal{F}(I) \cdot \mathcal{F}(h)|$ (log)", fontsize=14
)

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fft_img = np.fft.fft2(gray_img)
fft_filter = np.fft.fft2(np.fft.ifftshift(padded_filter))
fft_product = fft_img * fft_filter
ifft_result_real = np.real(np.fft.ifft2(fft_product))
ifft_result_real = np.clip(ifft_result_real, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(conv_spatial, cmap="gray")
axes[0].set_title("Spatial convolution")
axes[0].axis("off")

axes[1].imshow(ifft_result_real, cmap="gray")
axes[1].set_title("IFFT of fft_product")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
L = 11
kernel = np.ones((L, L), dtype=float) / (L * L)

padded_filter = np.zeros((H, W), dtype=float)
cy, cx = H // 2, W // 2

y_start = cy - L // 2
x_start = cx - L // 2
padded_filter[y_start : y_start + L, x_start : x_start + L] = kernel

filter_at_origin = np.fft.ifftshift(padded_filter)
fft_filter = np.fft.fft2(filter_at_origin)
real_spectrum = np.real(np.fft.fftshift(fft_filter))

power = 0.8
amp_power_scaled = np.power(np.abs(real_spectrum), power)
signed_spectrum = amp_power_scaled * np.sign(real_spectrum)

norm = TwoSlopeNorm(
    vmin=signed_spectrum.min(), vcenter=0, vmax=signed_spectrum.max()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].imshow(padded_filter, cmap="gray")
axes[0].set_title(f"Filter $h(x,y)$ (L={L})", fontsize=14)

axes[1].imshow(signed_spectrum, cmap="bwr", norm=norm)
axes[1].set_title(
    "Filter Spectrum $\mathcal{F}(h)$ (power scaled)\n(Red: +, Blue: -)",
    fontsize=14,
)

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
camera = cv2.imread(IMAGE_DATA_DIR / "camera.png", cv2.IMREAD_COLOR_RGB)

L = 11
box_filtered = cv2.boxFilter(camera, -1, (L, L), borderType=cv2.BORDER_REFLECT)
gauss_filtered = cv2.GaussianBlur(
    camera, (L, L), sigmaX=4, borderType=cv2.BORDER_REFLECT
)

y, x = 350, 250
h, w = 100, 100

crop_orig = camera[y : y + h, x : x + w]
crop_box = box_filtered[y : y + h, x : x + w]
crop_gauss = gauss_filtered[y : y + h, x : x + w]

fig, axes = plt.subplots(2, 3, figsize=(12, 10))

axes[0, 0].imshow(camera, cmap="gray")
axes[0, 0].set_title("Gaussian Filter")
axes[0, 0].axis("off")

axes[0, 1].imshow(box_filtered, cmap="gray")
axes[0, 1].set_title("Box Filter")
axes[0, 1].axis("off")

axes[0, 2].imshow(gauss_filtered, cmap="gray")
axes[0, 2].set_title("Gaussian Filter")
axes[0, 2].axis("off")

axes[1, 0].imshow(crop_orig, cmap="gray")
axes[1, 0].set_title("Box Filter (Zoom)")
axes[1, 0].axis("off")

axes[1, 1].imshow(crop_box, cmap="gray")
axes[1, 1].set_title("Box Filter (Zoom)")
axes[1, 1].axis("off")

axes[1, 2].imshow(crop_gauss, cmap="gray")
axes[1, 2].set_title("Gaussian Filter (Zoom)")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
gray_noisy = add_gaussian_noise(gray, sigma=15)

filtered_img, decision_map = classic_kuwahara(
    gray_noisy, radius=5, return_decision_map=True
)

y, x = 310, 180
h, w = 75, 75

gray_noisy_with_box = gray_noisy.copy()
cv2.rectangle(gray_noisy_with_box, (x, y), (x + w, y + h), 255, 2)

filtered_with_box = filtered_img.copy()
cv2.rectangle(filtered_with_box, (x, y), (x + w, y + h), 255, 2)

crop_noisy = gray_noisy[y : y + h, x : x + w]
crop_filter = filtered_img[y : y + h, x : x + w]
crop_decision = decision_map[y : y + h, x : x + w]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

axes[0][0].imshow(gray_noisy_with_box, cmap="gray")
axes[0][0].set_title("1. Noisy image", fontsize=14)

axes[0][1].imshow(filtered_with_box, cmap="gray")
axes[0][1].set_title("2. Filtered image", fontsize=14)

cmap_4 = ListedColormap(["#d62728", "#2ca02c", "#1f77b4", "#ff7f0e"])
axes[0][2].imshow(decision_map, cmap=cmap_4, interpolation="nearest")
axes[0][2].set_title("3. Decision map", fontsize=14)

# There is no white color in the palette, so we put rectangle on top
rect = patches.Rectangle(
    (x, y), w, h, linewidth=2, edgecolor="white", facecolor="none"
)
axes[0][2].add_patch(rect)

axes[1][0].imshow(crop_noisy, cmap="gray")
axes[1][0].set_title("Noisy image (Zoom)")

axes[1][1].imshow(crop_filter, cmap="gray")
axes[1][1].set_title("Filtered image (Zoom)")

axes[1][2].imshow(crop_decision, cmap=cmap_4)
axes[1][2].set_title("Decision map (Zoom)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
size = 300
Y, X = np.ogrid[:size, :size]
center = size / 2
dist_sq = (X - center) ** 2 + (Y - center) ** 2

sigma = 50
clean_circle = np.exp(-dist_sq / (2 * sigma**2))
clean_circle = (clean_circle * 255).astype(np.float32)

np.random.seed(42)
noisy_circle = add_gaussian_noise(clean_circle, sigma=2)

radius = 5
filtered_circle, decision_map = classic_kuwahara(
    clean_circle, radius=radius, return_decision_map=True
)
filtered_noisy_circle, noisy_decision_map = classic_kuwahara(
    noisy_circle, radius=radius, return_decision_map=True
)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(clean_circle, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("1. Gradient (clean)")

axes[0, 1].imshow(filtered_circle, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title(f"2. Kuwahara (r={radius})")

axes[0, 2].imshow(decision_map, cmap=cmap_4, interpolation="nearest")
axes[0, 2].set_title("3. Decision Map")

axes[1, 0].imshow(noisy_circle, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("4. Gradient (noisy $\sigma=2$)")

axes[1, 1].imshow(filtered_noisy_circle, cmap="gray", vmin=0, vmax=255)
axes[1, 1].set_title(f"5. Kuwahara (r={radius})")

axes[1, 2].imshow(noisy_decision_map, cmap=cmap_4, interpolation="nearest")
axes[1, 2].set_title("6. Decision Map")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()